In [78]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

In [79]:
path = "/content"

print("Path to competition files:", path)

Path to competition files: /content


In [80]:
train_df = pd.read_csv(path + "/train.csv")
test_df = pd.read_csv(path + "/test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (891, 12)
Test shape : (418, 11)


In [81]:
print(train_df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [82]:
X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

In [83]:

drop_columns = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
]

X = X.drop(columns=drop_columns)

In [84]:
numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

In [85]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)

X_train: (712, 7)
X_val  : (179, 7)


In [86]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [87]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)

print("\nShape before PCA:")
print("Train:", X_train_prepared.shape)
print("Validation:", X_val_prepared.shape)



Shape before PCA:
Train: (712, 12)
Validation: (179, 12)


In [88]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_prepared)
X_val_pca = pca.transform(X_val_prepared)

print("\nShape after PCA:")
print("Train:", X_train_pca.shape)
print("Validation:", X_val_pca.shape)

print("\nNumber of PCA components:", pca.n_components_)

print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)



Shape after PCA:
Train: (712, 7)
Validation: (179, 7)

Number of PCA components: 7
Explained variance: 0.9594052814197834


In [89]:
models = {

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Naive Bayes": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42

    )
}

In [90]:
def evaluate_model(model, X_train, X_val, y_train, y_val):


    start_train = time.perf_counter()

    model.fit(X_train, y_train)

    end_train = time.perf_counter()

    train_time = end_train - start_train

    start_predict = time.perf_counter()

    y_pred = model.predict(X_val)

    end_predict = time.perf_counter()

    predict_time = end_predict - start_predict

    accuracy = accuracy_score(y_val, y_pred)


    return accuracy, train_time, predict_time

In [91]:
results = []


In [92]:
#before pca
for name, model in models.items():

    accuracy, train_time, predict_time = evaluate_model(
        model,
        X_train_prepared,
        X_val_prepared,
        y_train,
        y_val
    )

    results.append({
        "Model": name,
        "PCA": "Before PCA",
        "Accuracy": accuracy,
        "Train Time (sec)": train_time,
        "Predict Time (sec)": predict_time
    })

    print(f"\n{name}")
    print("Accuracy     :", accuracy)
    print("Train Time   :", train_time)
    print("Predict Time :", predict_time)



KNN
Accuracy     : 0.8212290502793296
Train Time   : 0.0030029889999241277
Predict Time : 0.0052370230000633455

Naive Bayes
Accuracy     : 0.776536312849162
Train Time   : 0.0017266449999624456
Predict Time : 0.0003189799999745446

Decision Tree
Accuracy     : 0.8044692737430168
Train Time   : 0.003489469999976791
Predict Time : 0.00043082000001959386


In [93]:
#after pca
for name, model in models.items():

    accuracy, train_time, predict_time = evaluate_model(
        model,
        X_train_pca,
        X_val_pca,
        y_train,
        y_val
    )

    results.append({
        "Model": name,
        "PCA": "After PCA",
        "Accuracy": accuracy,
        "Train Time (sec)": train_time,
        "Predict Time (sec)": predict_time
    })

In [94]:
    print("Accuracy     :", accuracy)
    print("Train Time   :", train_time)
    print("Predict Time :", predict_time)

Accuracy     : 0.7653631284916201
Train Time   : 0.007893795999962094
Predict Time : 0.0003436229999351781


In [95]:
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

print(results_df)



FINAL RESULTS
           Model         PCA  Accuracy  Train Time (sec)  Predict Time (sec)
0            KNN  Before PCA  0.821229          0.003003            0.005237
1    Naive Bayes  Before PCA  0.776536          0.001727            0.000319
2  Decision Tree  Before PCA  0.804469          0.003489            0.000431
3            KNN   After PCA  0.793296          0.002610            0.004126
4    Naive Bayes   After PCA  0.793296          0.001149            0.000305
5  Decision Tree   After PCA  0.765363          0.007894            0.000344


In [96]:
print(
    results_df.sort_values(
        by="Accuracy",
        ascending=False
    )
)

           Model         PCA  Accuracy  Train Time (sec)  Predict Time (sec)
0            KNN  Before PCA  0.821229          0.003003            0.005237
2  Decision Tree  Before PCA  0.804469          0.003489            0.000431
4    Naive Bayes   After PCA  0.793296          0.001149            0.000305
3            KNN   After PCA  0.793296          0.002610            0.004126
1    Naive Bayes  Before PCA  0.776536          0.001727            0.000319
5  Decision Tree   After PCA  0.765363          0.007894            0.000344


In [97]:
for model in ["KNN", "Naive Bayes", "Decision Tree"]:

    before = results_df[
        (results_df["Model"] == model) &
        (results_df["PCA"] == "Before PCA")
    ].iloc[0]

    after = results_df[
        (results_df["Model"] == model) &
        (results_df["PCA"] == "After PCA")
    ].iloc[0]

    print("\n", model)
    print("Accuracy Before:", round(before["Accuracy"], 4))
    print("Accuracy After :", round(after["Accuracy"], 4))

    print("Train Time Before:", round(before["Train Time (sec)"], 5))
    print("Train Time After :", round(after["Train Time (sec)"], 5))


 KNN
Accuracy Before: 0.8212
Accuracy After : 0.7933
Train Time Before: 0.003
Train Time After : 0.00261

 Naive Bayes
Accuracy Before: 0.7765
Accuracy After : 0.7933
Train Time Before: 0.00173
Train Time After : 0.00115

 Decision Tree
Accuracy Before: 0.8045
Accuracy After : 0.7654
Train Time Before: 0.00349
Train Time After : 0.00789


In [98]:
results_df.to_csv(
    "titanic_models_pca_comparison.csv",
    index=False
)

print(
    "\nResults saved to titanic_models_pca_comparison.csv"
)


Results saved to titanic_models_pca_comparison.csv
